# 🧪 معمل التجارب: الفرق بين مخازن المتجهات وقواعد بيانات المتجهات (Lecture 24)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
في معمارية الـ RAG، أحد أهم القرارات الهندسية هو: أين سنخزن متجهاتنا الدلالية (Embeddings)؟
هل نستخدم **مخزن متجهات بسيط (Vector Store)** زي FAISS و Chroma يعمل مباشرة في رامات جهازك (In-Memory RAM)؟
أم نحتاج **قاعدة بيانات متجهات كاملة (Vector Database)** سحابية أو موزعة زي Pinecone, Qdrant, Milvus, Supabase (pgvector)؟

في هذا المعمل ستختبر بنفسك:
1. **سرعة البحث في الذاكرة (In-Memory Retrieval Benchmark):** استرجاع فائق السرعة خلال أجزاء من الميلي ثانية.
2. **التصفية بالبيانات الوصفية (Metadata Filtering):** تضييق نطاق البحث حسب تصنيف المستند.
3. **محرك اتخاذ القرار الهندسي:** أداة برمجية ذكية ترشح لك الحل الأنسب لمشروعك حسب حجم البيانات، الميزانية، وطبيعة التشغيل.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتجهيز التضمينات (Embeddings)
**الفكرة ببساطة:**
الـ Embeddings هي قلب أي نظام RAG، وظيفتها تحويل الجمل إلى مصفوفات أرقام تعكس معناها الحقيقي.
هنا بنستورد مكتبات المشروع وبنجهز نموذج التضمين المجاني من Hugging Face اللي بيشتغل محلياً على جهازك بسرعة وبدون أي تكلفة.

In [ ]:
# 1. import standard libraries and load environment variables
import os
import time
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# import langchain documents and vector store components
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# initialize embedding model (local huggingface embeddings)
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("✅ Libraries imported successfully!")
print("✅ HuggingFace Embedding model initialized and ready.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتجهيز نموذج التضمين بنفسك هنا):



### 2. تجربة مخزن المتجهات في الذاكرة (In-Memory) وقياس سرعة البحث
**الفكرة ببساطة:**
المخازن المضمنة زي FAISS بتشتغل داخل رامات جهازك مباشرة (Process Memory / RAM).
ميزتها الجبارة: البحث فيها سريع جداً وبياخد أجزاء من الميلي ثانية (أقل من 5 ميلي ثانية) لأن مفيش شبكة ولا سيرفرات خارجية في النص.
بنبني مجموعة مستندات ونجرب نبحث ونقيس الوقت بالظبط بـ `time.perf_counter()`.

In [ ]:
# sample corpus representing documents across domains with metadata
sample_corpus = [
    Document(page_content="LangChain provides standard interfaces for chains, agents, and retrieval.", metadata={"category": "ai", "tier": "free"}),
    Document(page_content="FAISS is an in-memory library for dense vector similarity search developed by Meta.", metadata={"category": "database", "tier": "open_source"}),
    Document(page_content="Pinecone is a managed cloud-native vector database designed for high availability at scale.", metadata={"category": "database", "tier": "enterprise"}),
    Document(page_content="PostgreSQL supports vector similarity search via the pgvector extension.", metadata={"category": "database", "tier": "open_source"}),
    Document(page_content="Qdrant is an open-source vector database written in Rust with advanced filtering.", metadata={"category": "database", "tier": "open_source"}),
]

# build in-memory FAISS vector store
vector_store = FAISS.from_documents(sample_corpus, embeddings)
print(f"✅ In-Memory Vector Store built with {vector_store.index.ntotal} vectors.\n")

# benchmark query retrieval latency
query = "What is an enterprise cloud vector database?"
start_time = time.perf_counter()
results = vector_store.similarity_search(query, k=2)
latency_ms = (time.perf_counter() - start_time) * 1000

print(f"⚡ Search Latency: {latency_ms:.3f} ms")
for rank, doc in enumerate(results, 1):
    print(f"  #{rank}: {doc.page_content} (Category: {doc.metadata['category']} | Tier: {doc.metadata['tier']})")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء مخزن FAISS في الذاكرة وقياس سرعة البحث بنفسك هنا):



### 3. البحث مع الفلترة بالبيانات الوصفية (Metadata Filtering)
**الفكرة ببساطة:**
تخيل عندك مليون وثيقة في الشركة، لكنك عايز تبحث فقط في الوثائق الخاصة بـ "قسم المبيعات" أو "الخطط المدفوعة (enterprise)".
عن طريق الـ Metadata Filtering، بنمرر شرط تصفية (مثلاً `tier == 'enterprise'`) فالمخزن يستبعد أي نتيجة تانية حتى لو كانت قريبة دلالياً.

In [ ]:
# filter query: only retrieve documents matching tier == 'enterprise'
filter_query = "database options"
enterprise_results = vector_store.similarity_search(
    filter_query,
    k=2,
    filter={"tier": "enterprise"}
)

print(f"🔍 Search Query: '{filter_query}' with filter tier == 'enterprise':\n")
for idx, doc in enumerate(enterprise_results, start=1):
    print(f"  [{idx}] Content: {doc.page_content}")
    print(f"      Metadata: {doc.metadata}\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل البحث مع تصفية Metadata من اختيارك هنا):



### 4. محرك اتخاذ القرار الهندسي (تختار إيه لمشروعك؟)
**الفكرة ببساطة:**
إزاي مهندس الـ AI المحترف بيختار بين FAISS, Chroma, Pinecone, Qdrant, أو Supabase (Postgres)؟
الموضوع مش صدفة، دي معادلة هندسية بتعتمد على:
- حجم البيانات (عدد المتجهات).
- هل المشروع تجريبي ولا Production حقيقي؟
- هل محتاج عمليات إضافة وحذف وتعديل لحظية (Real-time CRUD)؟
- والميزانية الشهرية المتوفرة.

كتبنا دالة برمجية ذكية `recommend_vector_storage` بتختبر سيناريوهات واقعية وترشح لك الحل الأنسب مع التعليل!

In [ ]:
def recommend_vector_storage(
    num_vectors: int,
    is_production: bool,
    needs_realtime_crud: bool,
    budget_usd_monthly: float
) -> dict:
    """Determine optimal vector storage architecture based on engineering constraints."""
    if not is_production or num_vectors < 50000:
        return {
            "Type": "Vector Store (Embedded)",
            "Recommendation": "FAISS or ChromaDB",
            "Rationale": "Dataset fits comfortably in memory. Zero cost, microsecond latency, zero infrastructure overhead."
        }
    
    if needs_realtime_crud and num_vectors >= 1000000:
        if budget_usd_monthly >= 70:
            return {
                "Type": "Vector Database (Managed Cloud)",
                "Recommendation": "Pinecone or Qdrant Cloud",
                "Rationale": "Full real-time CRUD, automatic horizontal scaling, high availability SLA, zero maintenance."
            }
        else:
            return {
                "Type": "Vector Database (Self-Hosted)",
                "Recommendation": "Qdrant (Docker) or Milvus / Pgvector",
                "Rationale": "Open-source enterprise DBMS. Full CRUD and scale without managed service recurring costs."
            }
            
    return {
        "Type": "Hybrid / Embedded Persistent",
        "Recommendation": "ChromaDB Persistent or Supabase (pgvector)",
        "Rationale": "Offers local persistence and simple updates without dedicated distributed cluster complexity."
    }

# test decision engine across realistic industry scenarios
scenarios = [
    ("Prototype / Academic RAG", 5000, False, False, 0),
    ("Production Knowledge Base (100K docs)", 100000, True, True, 20),
    ("Enterprise SaaS (10M real-time vectors)", 10000000, True, True, 300),
]

for name, count, prod, crud, budget in scenarios:
    rec = recommend_vector_storage(count, prod, crud, budget)
    print(f"📋 Scenario: {name}")
    print(f"  Category:       {rec['Type']}")
    print(f"  Recommendation: {rec['Recommendation']}")
    print(f"  Rationale:      {rec['Rationale']}\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب كتابة سيناريو مخصص وافحص ترشيح محرك القرار بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 24):
1. **المخزن المضمن (Vector Store):** زي FAISS و Chroma — سريع جداً (RAM)، مجاني، مثالي للنماذج الأولية والبيانات حتى مئات الآلاف من المتجهات.
2. **قاعدة بيانات المتجهات (Vector Database):** زي Pinecone و Qdrant و Milvus — تدعم مليارات المتجهات، وعمليات CRUD الحية، وتوزيع السيرفرات (Horizontal Scaling).
3. **الحل الهجين (Relational + Vector):** زي PostgreSQL مع `pgvector` و Supabase — ممتاز لما تكون محتاج بياناتك العادية مع المتجهات في قاعدة واحدة متكاملة!